In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 28.3 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Simple config
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SimpleDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),  # forces uniform size
            transforms.ToTensor()
        ])

        imgs = list(self.images_dir.glob('*.jpg'))
        self.pairs = [img for img in imgs
                     if (self.masks_dir / f"{img.stem}_mask.png").exists()]
        print(f" Dataset ready: {len(self.pairs)} pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path = self.pairs[idx]
        mask_path = self.masks_dir / f"{img_path.stem}_mask.png"

        # Load image (BGR→RGB)
        img = cv2.imread(str(img_path))[...,::-1]

        # Load mask and normalize
        mask = cv2.imread(str(mask_path), 0)
        mask = (mask > 127).astype(np.float32)  # Binary 0/1

        # Transform RESIZES everything to 128x128
        img = self.transform(img)
        mask = self.transform(mask)[0][None]  # 1x128x128

        return img, mask

# Simple CNN
class SimpleSegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1)
        )

    def forward(self, x):
        x = self.enc(x)
        x = self.dec(x)
        return torch.sigmoid(x)

# Setup
dataset = SimpleDataset(
    '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/train2017',
    '/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/mask_train2017'
)

train_loader = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=2)

model = SimpleSegNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

print(f"🚀 Training {len(dataset)} images")

# Training loop
for epoch in range(EPOCHS):
    model.train()
    loss_total = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for imgs, masks in pbar:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()

        loss_total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"Epoch {epoch+1} Avg Loss: {loss_total/len(train_loader):.4f}")

torch.save(model.state_dict(), 'simple_seg.pth')
print("✅ Saved model!")

# Test function
def remove_bg(img_path, model_path='simple_seg.pth'):
    model = SimpleSegNet()
    model.load_state_dict(torch.load(model_path))
    model.to(DEVICE).eval()

    img = cv2.imread(img_path)
    orig = img.copy()
    h, w = img.shape[:2]

    # Preprocess
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor()
    ])
    img_tensor = transform(img[...,::-1]).unsqueeze(0).to(DEVICE)

    # Predict
    with torch.no_grad():
        mask = model(img_tensor)[0,0].cpu().numpy()
        mask = cv2.resize(mask, (w, h)) > 0.5

    # Apply
    result = orig * mask[:,:,None]
    cv2.imwrite('result_no_bg.jpg', result)
    print("🎯 Saved: result_no_bg.jpg")

# Test
test_img = "/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE/train2017/000000532933.jpg"
remove_bg(test_img)


 Dataset ready: 3768 pairs
🚀 Training 3768 images


Epoch 1: 100%|██████████| 118/118 [45:56<00:00, 23.36s/it, loss=0.5866]


Epoch 1 Avg Loss: 0.5884


Epoch 2: 100%|██████████| 118/118 [02:58<00:00,  1.52s/it, loss=0.6405]


Epoch 2 Avg Loss: 0.5637


...

Epoch 8 Avg Loss: 0.5284


Epoch 9: 100%|██████████| 118/118 [03:02<00:00,  1.55s/it, loss=0.5289]


Epoch 9 Avg Loss: 0.5256


Epoch 10: 100%|██████████| 118/118 [03:01<00:00,  1.54s/it, loss=0.5513]


Epoch 10 Avg Loss: 0.5256
✅ Saved model!
🎯 Saved: result_no_bg.jpg
